In [7]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [4]:
# import json
import streamlit as st
import pdfplumber
import os

import faiss
import numpy as np
import tiktoken

from tqdm import tqdm


from dotenv import load_dotenv

from openai import OpenAI

In [3]:
load_dotenv()  # 加载 .env 文件
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


In [ ]:
# ---- Step 1: 加载原则文件 ----
with open("lucid_config.json", "r", encoding="utf-8") as f:
    principles = json.load(f)

# ---- Step 2: 把 JSON 转成自然语言（让 SYSTEM_PROMPT理解 ----
principles_text = "\n\n".join(
    [f"{k}:\n- " + "\n- ".join(v) for k, v in principles.items()]
)

### input = lucid_config.json

In [3]:

file_path = "./docs/PSI.pdf"
chunks = []

def generate_chunks(file_path, start_page=30, min_chars=100):
    with pdfplumber.open(file_path) as pdf:
        for page in pdf.pages[start_page-1:]:
            text = page.extract_text()
            if not text:
                continue
            paragraphs = text.split('\n\n')
            for para in paragraphs:
                if len(para.strip()) >= min_chars:
                    chunks.append(para.strip())
    print(f"✅ Total chunks: {len(chunks)}")
    return chunks



In [4]:
## embedding
def build_faiss_index(chunks, batch_size=100, embedding_model="text-embedding-3-small"):
    embeddings = []
    for start in tqdm(range(0, len(chunks), batch_size)):
        batch = chunks[start:start + batch_size]
        response = client.embeddings.create(
            input=batch,
            model=embedding_model
        )
        embeddings.extend([item.embedding for item in response.data])

    embeddings_array = np.array(embeddings).astype('float32')
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings_array)

    # 保存索引和 chunks
    faiss.write_index(index, "PSI.faiss")
    np.save("chunks.npy", chunks, allow_pickle=True)
    print("✅ 已保存索引和 chunks")
    return index


   

- print(embeddings[0]) 
-  显示 1536 个（或其他固定长度）浮点数字基于text-embedding-3-small"

### 每个 chunk embedded 后形成embedding，每个 embedding 是一个高维空间里的点，表示语义含义。 
- 不切 chunk，直接对全文做 embedding。粗。


In [5]:
def get_context_from_faiss(user_input, index_path="PSI.faiss", chunks_path="chunks.npy", k=15, embedding_model="text-embedding-3-small"):
    """
    输入用户 query，返回拼接好的 context 供 LLM 使用。
    
    参数:
    - query: str, 用户输入的问题
    - index_path: str, FAISS 索引文件路径
    - chunks_path: str, 保存 chunks 的文件路径
    - k: int, 检索最相关 chunk 数量
    - client: OpenAI client
    - embedding_model: str, embedding 模型
    """
    # 读取索引和 chunks
    index = faiss.read_index(index_path)
    chunks = np.load(chunks_path, allow_pickle=True)
    
    # 生成 query embedding
    user_input_emb = client.embeddings.create(
        input=user_input,
        model=embedding_model
    ).data[0].embedding
    
    # 检索最相关 chunk
    k = min(k, len(chunks))
    Distances, Indexes = index.search(np.array([user_input_emb]).astype('float32'), k=k)
    
    # 拼接 context，并添加文档引用
    context_parts = []
    for rank, (idx, dist) in enumerate(zip(Indexes[0], Distances[0]), start=1):
        chunk_text = chunks[idx]
        # 仅取前 N 个字符，避免过长
        snippet = chunk_text[:300] if len(chunk_text) > 300 else chunk_text
        context_parts.append(f"{snippet}\n\n[参考文档段落 {idx}]")
    
    context = "\n\n---\n\n".join(context_parts)
    return context


In [ ]:
def run_task_agent(user_input, messages=None, temperature=0.4, use_rag = True):
    '''
    messages
    包含 system prompt、用户输入和之前的 AI 输出，使 LLM 能够：
    记住行为规范（system）
    参考用户问题（user）
    结合之前回答和上下文（assistant）

    每轮对话，用户新输入追加到列表末尾，AI 输出追加到列表末尾。
    下一轮调用时，整个列表作为输入传给 LLM，保持历史记忆。
    '''

    # 初始化 messages 列表，不用每轮重复。
    if messages is None:
       developer_prompt = """你是一个游戏设置者。帮助用户以第三人称观察自己当前的任务状态。用户是游戏中的角色，目前遇到一个需要完成的任务。你的目标是通过提问和引导，让用户清晰地理解任务、分解步骤、设计可执行路径，并持续调整策略，帮助用户完成目标，不再因为清晰度不够而卡住拖延。
                            你不要直接给用户答案，而是：                            
                            1.                           
                            2.                             
                            3. 提出引导性的提问，让用户根据自己的目标和需求自己发现问题、反思目标与资源。                                                     
                            4. 保留用户历史输入，用于下一轮调整。                            
                            5. 结尾语气温暖，想一个思维清晰，态度真诚的朋友。
                            
                            """
    messages = [{"role": "developer", "content": developer_prompt}]
    
    # 添加用户新输入
    messages.append({"role": "user", "content": user_input})
    
    # 如果启用 RAG，则根据用户 query 检索 context
    if use_rag:
        context = get_context_from_faiss(user_input) # 此处调用get_context_from_faiss函数，生成 query embedding，匹配 vector DB，GET 30个 相关 chunks
        # 将 context 作为 system 补充信息加入 prompt
        messages.append({"role": "developer", "content": f"参考资料：\n{context}"})

    
    #获得llm输出
    assistant_output = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temperature
    ).choices[0].message.content
    
    # 保存 assistant 输出到 messages
    messages.append({"role": "assistant", "content": assistant_output})
    
    #打印
    print(assistant_output)
    
    return assistant_output, messages


In [5]:
##streamlit
st.set_page_config(page_title="RAG Agent Demo", page_icon="🤖")
st.title("🤖 RAG Agent Demo")


2025-10-30 09:24:29.721 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-30 09:24:29.723 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-30 09:24:29.837 
  command:

    streamlit run /Users/oooops/Desktop/Garden/.venv/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2025-10-30 09:24:29.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-10-30 09:24:29.838 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


DeltaGenerator()

In [11]:
if __name__ == "__main__":
    # 1. 生成 chunks 并构建索引（只需执行一次）
    faiss_path = "PSI.faiss"
    chunks_path = "chunks.npy"

    # 如果已存在缓存，则直接加载
    if os.path.exists(faiss_path) and os.path.exists(chunks_path):
        print("🧠 检测到已有缓存，直接加载索引和 chunks ...")
        index = faiss.read_index(faiss_path)
        chunks = np.load(chunks_path, allow_pickle=True).tolist()

    # 否则重新生成并缓存
    else:
        print("⚙️ 未检测到缓存，开始重新生成索引 ...")
        chunks = generate_chunks(file_path)
        index = build_faiss_index(chunks)
        print("✅ 索引生成完成并已保存")


    # 2. 命令行多轮查询
    # output_rag, _ = run_task_agent(query, use_rag=True)
    # output_no_rag, _ = run_task_agent("我想提高时间管理，但每次都拖延。", use_rag=False)
    while True:
        query = input("\n请输入问题 (exit退出): ")
        if query.lower() == "exit":
            break
        _, messages = run_task_agent(query, messages=messages)


🧠 检测到已有缓存，直接加载索引和 chunks ...


UnboundLocalError: cannot access local variable 'developer_prompt' where it is not associated with a value

In [47]:
# 第一轮 v2 引用
output, messages = run_task_agent("我想提高时间管理，但每次都拖延。",use_rag = True)

在这个故事中，用户正面临着一个挑战：他们希望提高时间管理的能力，但却总是陷入拖延之中。这个角色似乎在尝试设定目标和计划，但每次都被各种因素所干扰，导致无法有效执行。

从心理层面来看，用户可能正在经历以下几种状态：
- **思维模式**：可能存在对时间管理的期望过高，导致对自己施加过多压力，反而使得任务变得更加难以开始。
- **行为模式**：可能习惯于在最后一刻才开始工作，或者在面对任务时选择回避，而不是采取主动行动。
- **情绪状态**：可能感到焦虑或沮丧，因为未能如愿以偿地管理时间。
- **期望和假设**：用户可能假设只要设定了目标，就能自动执行，但实际上执行过程需要更多的策略和自我调节。

为了帮助用户更清晰地理解和应对这个挑战，可以提出以下引导性问题：
1. 你能否回忆起最近一次拖延的经历？当时你在想什么？你的情绪状态如何？
2. 你觉得是什么因素导致你在时间管理上感到困难？是外部环境的干扰，还是内心的焦虑？
3. 你有没有尝试过一些具体的时间管理技巧？如果有，它们的效果如何？
4. 设定目标时，你是否考虑到自己的能力和资源？是否有可能设定更小、更可行的目标？
5. 你是否有意识到，开始行动的第一步往往是最困难的？你可以尝试什么样的小步骤来克服这个障碍？

通过反思这些问题，用户可能会发现一些新的思路，帮助他们更好地理解自己在时间管理上的挑战，并找到适合自己的解决方案。记住，改变是一个过程，允许自己在这个过程中犯错和学习。

希望这些问题能帮助你更清晰地看待自己的时间管理目标，找到适合自己的路径。你并不孤单，很多人都在努力克服类似的挑战。保持耐心，继续前行！


In [ ]:
def external_brain(task: str, n_loops: int = 2):
    """
    让模型先回答，再自我反思并改进。
    n_loops 控制循环次数。
    """

    # Step 1: 初始思考
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}
        ],
        temperature=0.4
    )
    answer = response.choices[0].message.content.strip()

    # Step 2: 反思循环
    for i in range(n_loops):
        # 反思者
        reflection = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": "你是一个评论者，指出这段回答的不足和可改进之处，给出准确反馈。"},
                {"role": "user", "content": answer}
            ],
            temperature=0.4
        ).choices[0].message.content.strip()

        # 改进者
        improved = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "developer", "content": "根据下面的评论改进原回答。保持语言流畅，不重复评论内容。"},
                {"role": "user", "content": f"原回答：{answer}\n\n评论：{reflection}"}
            ],
            temperature=0.4
        ).choices[0].message.content.strip()

        answer = improved  # 更新为改进版本

    return answer





为了在没有恐惧的情况下与他人交往，我们需要从多个方面深入理解和建模。首先，识别恐惧的根源，其次，建立有效的沟通模型，最后，提升自我意识和自信心。以下是更具体的步骤：

1. **识别恐惧的来源**：
   - 恐惧往往源于对他人反应的预期、自我形象的担忧或社交场合的不确定性。通过自我反思，明确具体的恐惧点，如“我害怕被拒绝”或“我担心表达不清”。

   **具体方法**：记录在社交场合中让你感到不安的情境，并分析其中的关键因素。这一书写过程有助于清晰认识自己的恐惧，为后续应对策略奠定基础。

2. **建立有效的沟通模型**：
   - 理解交流的基本原则，如倾听、反馈和共情。有效沟通不仅是表达自己的想法，更是理解他人的需求和情感。

   **具体技巧**：在对话中使用开放式问题（如“你最近有什么新鲜事？”）来引导交流，减轻交流的压力，增强互动的自然性，促进双方理解。

3. **增强自我意识和自信心**：
   - 通过自我反思和积极自我对话，提升对自身价值的认知。认识到每个人都有优缺点，社交中并没有绝对的成功或失败。

   **具体示例**：在每次社交活动后，花几分钟回顾自己的表现，关注积极的方面，比如“我成功地与人交流”或“我表现得很放松”。这种积极反馈有助于逐步建立自信，并鼓励你在未来的社交中更加主动。

4. **实践与反馈**：
   - 在安全的环境中进行练习，例如与朋友或家人进行角色扮演，模拟社交场合。通过实践获得反馈，及时调整自己的沟通方式。

   **具体建议**：与朋友约定进行一次“社交练习”，在咖啡馆尝试与陌生人交谈。事后讨论彼此的感受和表现，互相提供建设性的反馈，并尝试在不同社交环境中进行练习，以帮助你适应各种情境。

5. **寻求情感支持**：
   - 面对社交恐惧时，情感支持和社交支持的作用不可忽视。与朋友、家人或专业人士分享感受，可以获得理解和鼓励。

   **具体做法**：建立一个支持网络，定期与信任的人交流进展和挑战，寻求他们的建议和鼓励。在选择支持对象时，优先考虑能够提供积极反馈和建设性建议的人。

6. **心理健康的考虑**：
   - 如果在面对社交恐惧时感到特别困难，寻求专业心理咨询可能是一个重要的支持途径。专业人士可以提供深入的指导和策略，帮助你有效应对社交焦虑。

通过这些步骤，你可以逐步减少社交中的恐

In [24]:
# 示例调用
task = "怎想到声音日记app。一方面，大家都做 app是各自的主动性/创意/能力实现；另一方面，觉得有点  “inefficient”...看作全球大脑，每个人是 neurons，如果合作，可拓展性很高？怎么 understand & evaluate 这个想法，"
output = external_brain(task, n_loops=3)
print(output)

要全面理解和评估“声音日记app”的构想，我们可以从多个关键方面进行深入分析和推理。

### 1. **明确问题与目标**
首先，需清晰界定该应用的核心目标。声音日记app旨在记录个人的声音日记，可能用于反思、生活记录或感受分享。我们应考虑以下问题：
- 该应用解决了哪些具体问题？
- 目标用户群体的特征是什么？例如，不同年龄段、性别和职业对用户需求的影响。

### 2. **市场分析与需求评估**
接下来，分析市场上现有的类似产品及其表现至关重要：
- 当前是否存在类似的声音记录应用？它们的优势与不足是什么？
- 用户对这些应用的反馈如何？是否存在未被满足的需求？引用具体的市场数据或用户调研结果将增强论点的说服力。

### 3. **全球大脑概念的深入应用**
提到的“全球大脑”概念可以帮助我们思考如何整合个体的声音记录，形成集体智慧：
- 如何设计这个应用以促进用户之间的有效合作？例如，是否可以实现声音日记的共享、评论和互动功能？
- 这种合作如何能够提升用户的反思能力和创造力？考虑具体的社交功能设计，帮助用户发现和关注其他用户的声音日记。

### 4. **效率与可扩展性分析**
关于“效率”问题，可以从以下几个方面进行深入探讨：
- 现有的声音记录方式是否真的存在低效？如果是，原因是什么？
- 如何利用云存储和AI技术来提升效率和可扩展性？提供一些具体的技术实现案例，说明这些技术将如何改善用户体验。

### 5. **反馈机制的具体设计**
建立有效的反馈机制对于不断优化应用功能至关重要：
- 如何系统地收集用户反馈？除了设置反馈按钮，还可以考虑使用NPS（净推荐值）调查、用户满意度调查等具体方法。
- 如何根据用户反馈调整应用功能，以更好地满足需求？可以探讨如何分析反馈数据以及进行A/B测试等。

### 6. **竞争分析**
在市场评估中，深入分析竞争对手及其优缺点是必要的：
- 列出主要竞争对手及其产品特性，分析它们的市场表现和用户反馈，以便更好地理解市场环境。

### 7. **创新功能与独特卖点**
在实际例子中，可以加入一些创新的功能或独特的卖点，以突出该应用的竞争优势：
- 考虑引入语音识别技术，将声音转化为文本，便于搜索和回顾，或设计个性化的声音日记模板，以满足不同用户的需求。

### 8. **商业模式与盈利策略**
讨论该

In [21]:
# ---- Step 4: 定义外置大脑函数（带自动反思） ----
def external_brain(task: str, reflection_rounds: int = 3):
    """
    执行一次主回答 + 可选的反思改进。
    reflect=True 时，自动让模型审视并改进自己的回答。
    """
    # 第一次回答（主思考）
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "developer", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task}
        ],
        temperature=0.4
    )
    output = response.choices[0].message.content.strip()

   

    # 多轮反思循环
    for i in range(reflection_rounds):
        reflect_prompt = f"""
    以下是你刚才的回答，请reflect 并 improve：
    1. 哪些部分只是通用建议或模板？
    2. 哪些地方未体现对“恐惧机制”或“表征模型”的理解？
    3. 给出具体例子帮助用户理解抽象语言了吗？ 
    4. 例子多样化吗，还是 biased单一方面的？
    5. 例子不少于 3 个，不多于 4 个。
    3. 请基于反思，输出改进版本，使内容更符合 rational external brain 的风格。
    
    
    以下是你上一次的输出：
    {output}
    ---
    请你反思其中的不足，并输出改进版本。
    只输出改进后的文本，不要说明理由。
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": reflect_prompt}
        ],
        temperature=0.4
    )

    improved_output = response.choices[0].message.content.strip()
    return improved_output


# ---- 使用示例 ----
if __name__ == "__main__":
    task = "怎么不带恐惧地与人交往？"
    output = external_brain(task)
    print(external_brain(task, reflection_rounds=3))
   



要不带恐惧地与人交往，首先需要深入理解恐惧的根源，并通过模型化来增强社交能力。以下是关键的思考步骤：

1. **识别恐惧的来源**：
   - 恐惧往往源于对他人评价的担忧、过去的负面经历或对社交情境的不确定性。通过反思这些来源，可以更清晰地理解自己的情绪。

   例如，如果你害怕在聚会上发言，回忆一下是否曾因发言而受到批评。这种具体的回忆可以帮助你识别出恐惧的根源。

2. **建立自我认知模型**：
   - 通过自我反思，建立对自己情绪和行为的模型。问自己：“我在社交中最担心的是什么？我希望别人如何看待我？”这有助于将情绪外化，便于分析。

   比如，写下你在社交场合中的常见想法，识别哪些是合理的，哪些是过度反应。你可能发现，担心别人觉得你无聊其实并没有根据。

3. **设定小目标**：
   - 从小的社交互动开始，逐步扩大范围。设定具体的、可实现的目标，例如在一次聚会上主动与一个人交谈，而不是试图与所有人交流。

   例如，你可以选择在工作中与一个同事共进午餐，而不是参加大型的社交活动，以此逐步建立信心。

4. **练习积极的自我对话**：
   - 用积极的语言与自己对话，替代消极的自我评价。当感到紧张时，可以告诉自己：“我有能力与他人交流，我的观点是有价值的。”

   你可以在镜子前练习自我对话，增强自信心，比如在准备出门前，告诉自己“我会享受这次交流”。

5. **关注对方而非自己**：
   - 在社交互动中，将注意力从自己的恐惧转移到对方身上。倾听对方的故事和感受，建立真正的连接。

   例如，在与他人交谈时，专注于他们的表情和反应，而不是在心中反复思考自己的表现，试着问对方问题，了解他们的兴趣。

6. **接受不完美**：
   - 理解每个人都有自己的不安和缺点，接受自己的不完美是社交的一部分。将社交视为一种学习和成长的机会，而不是一个完美表现的舞台。

   比如，回顾一次社交经历，即使没有达到预期，也要庆祝自己迈出了这一步，意识到每次尝试都是进步。

通过以上步骤，你可以逐步降低与人交往时的恐惧感，增强自己的社交能力。记住，社交是一种技能，随着实践和反思会不断提高。


In [ ]:
import re
import numpy as np
import faiss
from tqdm import tqdm
import pdfplumber

class ExternalBrainAgent:
    def __init__(self, client, embedding_model="text-embedding-3-small", index_file="agent_index.faiss", chunks_file="agent_chunks.npy"):
        """
        client: OpenAI client
        embedding_model: embedding 模型
        index_file: FAISS index 保存路径
        chunks_file: chunk 保存路径
        """
        self.client = client
        self.embedding_model = embedding_model
        self.index_file = index_file
        self.chunks_file = chunks_file
        self.chunks = []  # [(source, chunk_text)]
        self.index = None

    # --------------------------
    # 文档处理
    # --------------------------
    def load_pdf(self, file_path):
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def chunk_text(self, text, source, min_size=300, max_size=800):
        """
        source: 标记来源，例如 PERSONAL, THINKING_BOOK, OTHER
        """
        paragraphs = re.split(r'\n\s*\n+', text)
        chunks = []
        current_chunk = ""

        for para in paragraphs:
            para = para.strip()
            if not para:
                continue
            if len(current_chunk) + len(para) < max_size:
                current_chunk += "\n\n" + para if current_chunk else para
            else:
                if len(current_chunk) >= min_size:
                    chunks.append((source, current_chunk))
                current_chunk = para
        if current_chunk and len(current_chunk) >= min_size:
            chunks.append((source, current_chunk))

        self.chunks.extend(chunks)
        return chunks

    # --------------------------
    # Embedding + FAISS
    # --------------------------
    def build_index(self, batch_size=100):
        embeddings = []
        for start in tqdm(range(0, len(self.chunks), batch_size)):
            batch = [chunk_text for _, chunk_text in self.chunks[start:start+batch_size]]
            response = self.client.embeddings.create(input=batch, model=self.embedding_model)
            embeddings.extend([item.embedding for item in response.data])
        
        embeddings_array = np.array(embeddings).astype("float32")
        dim = embeddings_array.shape[1]
        self.index = faiss.IndexFlatL2(dim)
        self.index.add(embeddings_array)

        # 保存索引和 chunks
        faiss.write_index(self.index, self.index_file)
        np.save(self.chunks_file, self.chunks, allow_pickle=True)
        print("✅ 已保存索引和 chunks")

    def load_index(self):
        self.index = faiss.read_index(self.index_file)
        self.chunks = np.load(self.chunks_file, allow_pickle=True).tolist()
        print(f"✅ 已加载 {len(self.chunks)} chunks 和索引")

    # --------------------------
    # 查询与生成答案
    # --------------------------
    def query(self, query_text, top_k=20, temperature=0.2, system_prompt=None):
        # embed query
        query_emb = self.client.embeddings.create(
            input=query_text,
            model=self.embedding_model
        ).data[0].embedding

        # search
        D, I = self.index.search(np.array([query_emb]).astype("float32"), k=min(top_k, len(self.chunks)))

        # 拼接 context，保留来源
        context_parts = []
        for idx in I[0]:
            source, chunk_text = self.chunks[idx]
            context_parts.append(f"[{source}]\n{chunk_text}")
        context = "\n\n---\n\n".join(context_parts)

        # LLM 生成答案
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": f"以下是相关信息：\n\n{context}\n\n用户问题：{query_text}"})

        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=temperature
        )
        return response.choices[0].message.content

# --------------------------
# 使用示例
# --------------------------
# agent = ExternalBrainAgent(client)
# text_diary = agent.load_pdf("./docs/personal.pdf")
# agent.chunk_text(text_diary, source="PERSONAL")
# text_book = agent.load_pdf("./docs/agent_thinking_book.pdf")
# agent.chunk_text(text_book, source="THINKING_BOOK")
# agent.build_index()

# # 查询
# system_prompt = "请根据个人信息提供个性化建议，并遵循书籍的思考方法。"
# answer = agent.query("如何提高我的 task completion 能力？", top_k=20, system_prompt=system_prompt)
# print(answer)
